<a href="https://colab.research.google.com/github/miriamamin1213-ux/HPV-classification/blob/main/Model27.11F.43.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# MODEL 27
# Conv16-FC32-FC16-2
# ============================================================

!pip install -q gdown

import gdown

gdown.download(
    id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv",
    output="HPV2025.xlsx",
    quiet=False
)

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score
)

from imblearn.over_sampling import SMOTE

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

def seed_everything(seed=43):

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    np.random.seed(seed)
    random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    torch.use_deterministic_algorithms(True, warn_only=True)

    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False

SEED=43
seed_everything(SEED)

# ------------------------------------------------------------
# Read Dataset
# ------------------------------------------------------------

df=pd.read_excel("HPV2025.xlsx")

df=df.dropna(subset=["HPV Status"])

tobacco_mode=df["Tobacco Consumption"].mode()[0]

df["Tobacco Consumption"]=df[
    "Tobacco Consumption"
].fillna(tobacco_mode)

alcohol_mode=df["Alcohol Consumption"].mode()[0]

df["Alcohol Consumption"]=df[
    "Alcohol Consumption"
].fillna(alcohol_mode)

df=df.drop(
    columns=[
        "PatientID",
        "CenterID",
        "Task 1",
        "Task 2",
        "Task 3"
    ]
)

df=df.dropna()

print(df.shape)

# ------------------------------------------------------------
# Encode stages
# ------------------------------------------------------------

df["T-stage"]=df["T-stage"].replace({
"T0":0,
"T1":1,
"T2":2,
"T3":3,
"T4":4
})

df["N-stage"]=df["N-stage"].replace({
"N0":0,
"N1":1,
"N2":2,
"N3":3
})

df["M-stage"]=df["M-stage"].replace({
"M0":0,
"M1":1
})

# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

X=df[
[
"Age",
"Gender",
"Tobacco Consumption",
"Alcohol Consumption",
"Performance Status",
"Relapse",
"RFS",
"Treatment",
"T-stage",
"N-stage",
"M-stage"
]
]

y=df["HPV Status"]

# ------------------------------------------------------------
# Split
# ------------------------------------------------------------

X_train,X_test,y_train,y_test=train_test_split(

X,
y,

test_size=0.20,

random_state=SEED

)

print(y_train.value_counts())

print(y_test.value_counts())

# ------------------------------------------------------------
# Standardisation
# ------------------------------------------------------------

scaler=StandardScaler()

X_train=scaler.fit_transform(X_train)

X_test=scaler.transform(X_test)

# ------------------------------------------------------------
# SMOTE
# ------------------------------------------------------------

smote=SMOTE(random_state=SEED)

X_train_smote,y_train_smote=smote.fit_resample(

X_train,
y_train

)

print(pd.Series(y_train_smote).value_counts())

# ------------------------------------------------------------
# Torch
# ------------------------------------------------------------

X_train_smote=torch.FloatTensor(X_train_smote)

X_test=torch.FloatTensor(X_test)

y_train_smote=torch.LongTensor(
y_train_smote.to_numpy())

y_test=torch.LongTensor(
y_test.to_numpy())

# Conv1D requires channel dimension

X_train_smote=X_train_smote.unsqueeze(1)

X_test=X_test.unsqueeze(1)

# ------------------------------------------------------------
# MODEL 27
# Conv16-FC32-FC16-FC2
# ------------------------------------------------------------

class HPVCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1=nn.Conv1d(

            in_channels=1,

            out_channels=16,

            kernel_size=3,

            padding=1

        )

        self.relu=nn.ReLU()

        self.flatten=nn.Flatten()

        self.fc1=nn.Linear(
            16*11,
            32
        )

        self.fc2=nn.Linear(
            32,
            16
        )

        self.fc3=nn.Linear(
            16,
            2
        )

    def forward(self,x):

        x=self.conv1(x)

        x=self.relu(x)

        x=self.flatten(x)

        x=self.relu(
            self.fc1(x)
        )

        x=self.relu(
            self.fc2(x)
        )

        x=self.fc3(x)

        return x

model=HPVCNN()

weights=torch.tensor(
    [2.0,1.0],
    dtype=torch.float32
)

criterion=nn.CrossEntropyLoss(
    weight=weights
)

optimizer=torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

# ------------------------------------------------------------
# TRAINING
# ------------------------------------------------------------

epochs=500

best_test_loss=float("inf")

best_epoch=0

for epoch in range(epochs):

    model.train()

    outputs=model(X_train_smote)

    train_loss=criterion(
        outputs,
        y_train_smote
    )

    optimizer.zero_grad()

    train_loss.backward()

    optimizer.step()

    model.eval()

    with torch.no_grad():

        test_outputs=model(X_test)

        test_loss=criterion(
            test_outputs,
            y_test
        )

    if test_loss.item()<best_test_loss:

        best_test_loss=test_loss.item()

        best_epoch=epoch+1

        torch.save(
            model.state_dict(),
            "best_model27.pth"
        )

    if (epoch+1)%50==0:

        print(

            f"Epoch {epoch+1}, "

            f"Train={train_loss.item():.4f}, "

            f"Test={test_loss.item():.4f}"

        )

print()

print("Best Test Loss =",round(best_test_loss,4))

print("Best Epoch =",best_epoch)

# ------------------------------------------------------------
# LOAD BEST MODEL
# ------------------------------------------------------------

model.load_state_dict(
    torch.load("best_model27.pth")
)

model.eval()

with torch.no_grad():

    outputs=model(X_test)

    probabilities=torch.softmax(
        outputs,
        dim=1
    )

    predicted=torch.argmax(
        outputs,
        dim=1
    )

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print()

print("Classification Report\n")

print(

classification_report(

y_test.numpy(),

predicted.numpy(),

digits=4

)

)

# ------------------------------------------------------------
# CONFUSION MATRIX
# ------------------------------------------------------------

cm=confusion_matrix(

y_test.numpy(),

predicted.numpy()

)

print("Confusion Matrix")

print(cm)

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

bal_acc=balanced_accuracy_score(

y_test.numpy(),

predicted.numpy()

)

f1=f1_score(

y_test.numpy(),

predicted.numpy()

)

auc=roc_auc_score(

y_test.numpy(),

probabilities[:,1].numpy()

)

accuracy=(

predicted.eq(y_test)

.sum()

.item()

/

len(y_test)

)

print()

print(f"Accuracy:            {accuracy:.4f}")

print(f"Balanced Accuracy:   {bal_acc:.4f}")

print(f"F1-score:            {f1:.4f}")

print(f"AUC:                 {auc:.4f}")

# ------------------------------------------------------------
# MODEL SUMMARY
# ------------------------------------------------------------

print()

print("====================================")

print("MODEL 27 SUMMARY")

print("====================================")

print("Architecture : Conv16-FC32-FC16-2")

print("Input Features : 11")

print("Learning Rate : 0.0001")

print("Optimiser : Adam")

print("Class Weights : [2.0,1.0]")

print("SMOTE : Yes")

print("Epochs : 500")

print("Best Epoch :",best_epoch)

print("Best Test Loss :",round(best_test_loss,4))

print("Training Patients :",len(y_train))

print("Test Patients :",len(y_test))

print("Training after SMOTE :",len(y_train_smote))

print("Seed :",SEED)

Downloading...
From: https://drive.google.com/uc?id=1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv
To: /content/HPV2025.xlsx
100%|██████████| 66.0k/66.0k [00:00<00:00, 2.07MB/s]


(423, 12)
HPV Status
1.0    321
0.0     17
Name: count, dtype: int64
HPV Status
1.0    79
0.0     6
Name: count, dtype: int64
HPV Status
1.0    321
0.0    321
Name: count, dtype: int64


/tmp/ipykernel_1065/2960246741.py:100: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["T-stage"]=df["T-stage"].replace({
/tmp/ipykernel_1065/2960246741.py:108: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["N-stage"]=df["N-stage"].replace({
/tmp/ipykernel_1065/2960246741.py:115: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_sil

Epoch 50, Train=0.6818, Test=0.6264
Epoch 100, Train=0.6175, Test=0.6162
Epoch 150, Train=0.5278, Test=0.5666
Epoch 200, Train=0.4356, Test=0.5169
Epoch 250, Train=0.3591, Test=0.4733
Epoch 300, Train=0.3031, Test=0.4419
Epoch 350, Train=0.2643, Test=0.4167
Epoch 400, Train=0.2368, Test=0.4003
Epoch 450, Train=0.2164, Test=0.3867
Epoch 500, Train=0.2005, Test=0.3770

Best Test Loss = 0.377
Best Epoch = 500

Classification Report

              precision    recall  f1-score   support

           0     0.3158    1.0000    0.4800         6
           1     1.0000    0.8354    0.9103        79

    accuracy                         0.8471        85
   macro avg     0.6579    0.9177    0.6952        85
weighted avg     0.9517    0.8471    0.8800        85

Confusion Matrix
[[ 6  0]
 [13 66]]

Accuracy:            0.8471
Balanced Accuracy:   0.9177
F1-score:            0.9103
AUC:                 0.9093

MODEL 27 SUMMARY
Architecture : Conv16-FC32-FC16-2
Input Features : 11
Learning Rate : 0.